# Module 02 Lab — Multi-Tool Agent with Sandboxed Code Runner

**Course:** AI Agents Teaching Platform  
**Module:** 02 — Tools & Function Calling  
**Estimated time:** 2–3 hours

---

### What you'll build

A working agent with three tools: web search, scoped file access, and a sandboxed code runner.
Every tool handles errors gracefully and has a JSON schema that rejects invalid input.

### Hard constraints
- File tool must reject paths outside `./workspace/`
- Code runner must time out after 10 seconds
- No unhandled Python exceptions — all errors return structured error dicts
- MAX_STEPS ≤ 15

## Step 1 — Choose your model

| Provider | Model string | Key env var |
|---|---|---|
| Anthropic | `claude-haiku-4-5-20251001` | `ANTHROPIC_API_KEY` |
| OpenAI | `gpt-4o-mini` | `OPENAI_API_KEY` |
| Google Gemini | `gemini/gemini-1.5-flash` | `GEMINI_API_KEY` |
| Groq | `groq/llama-3.1-8b-instant` | `GROQ_API_KEY` |

In [ ]:
MODEL = "claude-haiku-4-5-20251001"
print(f"Model: {MODEL}")

In [ ]:
%pip install -q litellm httpx jsonschema

In [ ]:
import os
from getpass import getpass

os.environ["ANTHROPIC_API_KEY"] = getpass("Paste your API key: ")
print("Key set ✓")

# Create workspace directory
os.makedirs("./workspace", exist_ok=True)
print("Workspace created ✓")

## Step 4 — Define tool schemas (TODO)

The `search_web` schema is complete. Fill in `READ_FILE_SCHEMA` and `RUN_CODE_SCHEMA`.

Each schema needs:
- `name` matching the function name
- `description` with a 'when NOT to use' clause
- `parameters` with types, descriptions, and `required` fields

Note: LiteLLM uses the same OpenAI tool schema format.

In [ ]:
SEARCH_SCHEMA = {
    "type": "function",
    "function": {
        "name": "search_web",
        "description": "Search the web. Do NOT use for questions answerable from general knowledge.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Specific search query"},
                "max_results": {"type": "integer", "minimum": 1, "maximum": 10, "default": 5}
            },
            "required": ["query"]
        }
    }
}

# TODO: fill in these two schemas
READ_FILE_SCHEMA = {
    # name: "read_file"
    # description: what it reads, when NOT to use it (don't use for URLs)
    # parameters: path (string, required)
}

RUN_CODE_SCHEMA = {
    # name: "run_code"
    # description: executes Python in a subprocess, mention the 10s timeout
    # parameters: code (string, required), timeout (integer, min 1, max 10, default 10)
}

TOOLS = [SEARCH_SCHEMA, READ_FILE_SCHEMA, RUN_CODE_SCHEMA]
print("Schemas defined")

<details>
<summary>💡 Stuck? Reveal the schema solutions</summary>

```python
READ_FILE_SCHEMA = {
    "type": "function",
    "function": {
        "name": "read_file",
        "description": "Read a text file from the agent workspace. Only files inside workspace are accessible. Do NOT use for URLs — use search_web.",
        "parameters": {
            "type": "object",
            "properties": {
                "path": {"type": "string", "description": "Path relative to workspace, e.g. 'notes/summary.txt'"}
            },
            "required": ["path"]
        }
    }
}

RUN_CODE_SCHEMA = {
    "type": "function",
    "function": {
        "name": "run_code",
        "description": "Execute a short Python script in a sandboxed subprocess. Scripts are killed after 10s — do NOT use for long-running programs.",
        "parameters": {
            "type": "object",
            "properties": {
                "code": {"type": "string", "description": "Complete Python source to execute"},
                "timeout": {"type": "integer", "minimum": 1, "maximum": 10, "default": 10}
            },
            "required": ["code"]
        }
    }
}
```

</details>

## Step 5 — Implement the tools (TODO)

Fill in all three tool functions. Key requirements:
- `search_web`: use httpx, handle `TimeoutException` and `HTTPStatusError`, return `{"results": [...]}` or `{"error": ..., "retryable": bool}`
- `read_file`: use `os.path.realpath` to resolve the path, check it starts with `WORKSPACE`, return `{"content": ...}` or `{"error": ...}`
- `run_code`: use `subprocess.run` with `timeout=min(timeout, 10)`, handle `TimeoutExpired`

In [ ]:
import subprocess
import httpx

WORKSPACE = os.path.realpath("./workspace")

def search_web(query: str, max_results: int = 5) -> dict:
    """TODO: call DuckDuckGo API with httpx, handle errors, return {results: [...]}."""
    # Hint: httpx.get("https://api.duckduckgo.com/",
    #                 params={"q": query, "format": "json", "no_html": 1}, timeout=10)
    pass

def read_file(path: str) -> dict:
    """TODO: resolve path with os.path.realpath(os.path.join(WORKSPACE, path)),
    check it starts with WORKSPACE, return {content: ...} or {error: ...}."""
    pass

def run_code(code: str, timeout: int = 10) -> dict:
    """TODO: subprocess.run(["python", "-c", code], cwd=WORKSPACE,
    capture_output=True, text=True, timeout=min(timeout, 10)).
    Handle TimeoutExpired. Return {stdout, stderr, returncode}."""
    pass

TOOL_MAP = {"search_web": search_web, "read_file": read_file, "run_code": run_code}
print("Tool functions defined")

<details>
<summary>💡 Stuck? Reveal the tool implementations</summary>

```python
def search_web(query: str, max_results: int = 5) -> dict:
    try:
        resp = httpx.get("https://api.duckduckgo.com/",
                         params={"q": query, "format": "json", "no_html": 1}, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        results = [
            {"title": t.get("Text", ""), "url": t.get("FirstURL", "")}
            for t in data.get("RelatedTopics", [])[:max_results]
            if isinstance(t, dict) and t.get("FirstURL")
        ]
        return {"results": results}
    except httpx.TimeoutException:
        return {"error": "TimeoutError", "message": "Search timed out", "retryable": True}
    except httpx.HTTPStatusError as e:
        return {"error": "HTTPError", "message": str(e), "retryable": e.response.status_code >= 500}

def read_file(path: str) -> dict:
    resolved = os.path.realpath(os.path.join(WORKSPACE, path))
    if not resolved.startswith(WORKSPACE + os.sep) and resolved != WORKSPACE:
        return {"error": "PathError", "message": "Path outside workspace", "retryable": False}
    try:
        with open(resolved, "r", encoding="utf-8") as f:
            return {"content": f.read(20_000)}
    except FileNotFoundError:
        return {"error": "NotFound", "message": f"No such file: {path}", "retryable": False}

def run_code(code: str, timeout: int = 10) -> dict:
    timeout = min(timeout, 10)
    try:
        result = subprocess.run(["python", "-c", code],
                                cwd=WORKSPACE, capture_output=True, text=True, timeout=timeout)
        return {"stdout": result.stdout[-5000:], "stderr": result.stderr[-5000:],
                "returncode": result.returncode}
    except subprocess.TimeoutExpired:
        return {"error": "TimeoutError", "message": f"Script exceeded {timeout}s", "retryable": False}
```

</details>

## Step 6 — The agent loop

The loop is mostly complete. The one TODO: add argument validation before executing a tool.
Use `jsonschema.validate(args, schema["function"]["parameters"])` and catch `ValidationError`.

In [ ]:
import json
import litellm
import jsonschema

litellm.drop_params = True
MAX_STEPS = 10


def run_agent(user_message: str) -> str:
    messages = [{"role": "user", "content": user_message}]

    for step in range(MAX_STEPS):
        response = litellm.completion(model=MODEL, messages=messages, tools=TOOLS)
        msg = response.choices[0].message
        messages.append({"role": "assistant", "content": msg.content,
                          "tool_calls": [tc.dict() for tc in (msg.tool_calls or [])]})

        if not msg.tool_calls:
            return msg.content

        for call in msg.tool_calls:
            name = call.function.name
            try:
                args = json.loads(call.function.arguments)
            except json.JSONDecodeError:
                result = {"error": "BadArguments", "message": "Arguments were not valid JSON"}
            else:
                if name not in TOOL_MAP:
                    result = {"error": "UnknownTool", "message": f"No such tool: {name}"}
                else:
                    # TODO: validate args against schema before calling TOOL_MAP[name](**args)
                    # Use jsonschema.validate and catch jsonschema.ValidationError
                    result = TOOL_MAP[name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": call.id,
                "content": json.dumps(result)
            })

    return "Agent reached maximum steps without completing."


print("Agent loop defined ✓")

## Step 7 — Run the tests

In [ ]:
# Test 1: path traversal must be blocked
result = read_file("../../etc/passwd")
assert "error" in result, "Path traversal must be blocked"
print("Test 1 (path traversal): PASS ✓")

In [ ]:
# Test 2: infinite loop must time out
result = run_code("while True: pass", timeout=2)
assert result.get("error") == "TimeoutError", f"Expected TimeoutError, got {result}"
print("Test 2 (timeout): PASS ✓")

In [ ]:
# Test 3: full agent run
output = run_agent("What is 7 * 8? Save the answer to workspace/math.txt then read it back.")
print(f"Agent output: {output}")
assert isinstance(output, str)
print("Test 3 (full run): PASS ✓")

## Step 8 — Experiments

### Experiment A: Blast radius
Try `run_code("import shutil; shutil.rmtree('/tmp')")`. What happens? (The `cwd=WORKSPACE` scope limits damage, but note what the code runner *can* do.)

### Experiment B: Schema rejection
Call `search_web(query="", max_results=100)`. The `max_results` schema has `maximum: 10`. What does jsonschema say?

### Experiment C: retryable flag
In `run_agent`, add logic to retry a tool call if `result.get("retryable") == True`. How many times should you retry before giving up?

### Experiment D: Swap the model
Change `MODEL` to `gpt-4o-mini` and re-run Test 3. Does the agent still complete the task?

In [ ]:
# Scratch cell — use for experiments


## Step 9 — Reflection

1. Name the blast radius of each tool and how you bounded it.
2. Why does `read_file` use `os.path.realpath` instead of just checking for `../`?
3. What does `retryable: True` in an error response mean for the agent loop?
4. In Experiment D, what was the only line you changed to switch providers?

*(Double-click to edit)*

1. 
2. 
3. 
4. 